<a href="https://colab.research.google.com/github/sakshimohta17/TechMedBuddy/blob/main/BLAST_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install biopython

from Bio import SeqIO
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.Seq import Seq
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.1 MB/s eta 0:00:00


In [4]:
def run_blast(sequence, program="blastn", database="nt", expect=10.0,
              hitlist_size=10, entrez_query=None, word_size=None,
              matrix_name=None, gapcosts=None, filter=None):
    """
    Runs a BLAST search against an NCBI database.

    Args:
        sequence: The nucleotide or protein sequence to search with.
        program: blastn, blastp, blastx, tblastn, or tblastx. Defaults to blastn.
        database: "nt" for nucleotide databases, "nr" for protein databases.
        expect: E-value threshold for reporting hits. Defaults to 10.0.
        hitlist_size: Maximum number of hits to return. Defaults to 10.
        entrez_query: Entrez query to limit results (e.g. "Homo sapiens[organism]").
        word_size: Word size for the initial seed match.
        matrix_name: Scoring matrix (e.g. "BLOSUM62") for protein searches.
        gapcosts: Gap opening and extension costs, e.g. "11 1".
        filter: Low-complexity filter setting.

    Returns:
        A list of BLAST records.
    """
    print(f"Running {program} against {database}: {sequence[:50]}...")
    result_handle = NCBIWWW.qblast(program, database, sequence, expect=expect,
                                   hitlist_size=hitlist_size, entrez_query=entrez_query,
                                   word_size=word_size, matrix_name=matrix_name,
                                   gapcosts=gapcosts, filter=filter)
    records = list(NCBIXML.parse(result_handle))
    result_handle.close()
    return records

In [5]:
from google.colab import files

uploaded = files.upload()
fasta_file = list(uploaded.keys())[0]

def read_fasta(fasta_file):
    """Returns the first sequence in a FASTA file."""
    with open(fasta_file, "r") as handle:
        record = next(SeqIO.parse(handle, "fasta"))
        return str(record.seq)

sequence_dna = read_fasta(fasta_file)

# Translate frame 1 up to the first stop codon for the protein-query searches
sequence_protein = str(Seq(sequence_dna).translate(to_stop=True))

print(f"DNA length: {len(sequence_dna)} bp")
print(f"Protein length: {len(sequence_protein)} aa")

Saving gene.fna to gene.fna
DNA length: 19070 bp
Protein length: 31 aa


/usr/local/lib/python3.13/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [ ]:
# Each search: (query sequence, database)
# nt = nucleotide database, nr = protein database
searches = {
    "blastn":  (sequence_dna,     "nt"),  # DNA vs DNA
    "blastx":  (sequence_dna,     "nr"),  # translated DNA vs protein
    "blastp":  (sequence_protein, "nr"),  # protein vs protein
    "tblastn": (sequence_protein, "nt"),  # protein vs translated DNA
    "tblastx": (sequence_dna,     "nt"),  # translated DNA vs translated DNA
}

blast_results = {}
for program, (query, db) in searches.items():
    try:
        blast_results[program] = run_blast(query, program=program, database=db)
    except Exception as e:
        print(f"{program} failed: {e}")
        blast_results[program] = []

Running blastn against nt: CTCAAAAGTCTAGAGCCACCGTCCAGGGAGCAGGTAGCTGCTGGGCTCCG...
Running blastx against nr: CTCAAAAGTCTAGAGCCACCGTCCAGGGAGCAGGTAGCTGCTGGGCTCCG...


In [ ]:
rows = []
for program, records in blast_results.items():
    for record in records:
        for alignment in record.alignments:
            hsp = alignment.hsps[0]  # best-scoring segment for this hit
            rows.append({
                "program": program,
                "hit": alignment.title[:100],
                "accession": alignment.accession,
                "length": alignment.length,
                "e_value": hsp.expect,
                "score": hsp.score,
                "identity_%": round(100 * hsp.identities / hsp.align_length, 2),
            })

results_df = pd.DataFrame(rows)
display(results_df.sort_values(["program", "e_value"]).groupby("program").head(5))

results_df.to_csv("blast_results.csv", index=False)
files.download("blast_results.csv")